# Derivatives: Single-Sided vs. Central Difference Formulations

This notebook examines the mathematical and numerical mechanics of **differentiation**, the engine that powers backpropagation and gradient descent in neural networks.

## Mathematical Definitions

For a continuous function $f: \mathbb{R} \to \mathbb{R}$, the derivative at point $x$ is defined as the limit of the difference quotient:

$$f'(x) = \lim_{h \to 0} \frac{f(x + h) - f(x)}{h}$$

In computational environments where exact symbolic derivatives are unavailable (such as black-box simulation or gradient checking), we approximate $f'(x)$ using finite differences:

1. **Forward Difference (One-Sided):**
   $$D_{\text{forward}}(f, x, h) = \frac{f(x + h) - f(x)}{h}$$

2. **Central Difference:**
   $$D_{\text{central}}(f, x, h) = \frac{f(x + h/2) - f(x - h/2)}{h} \quad \text{or} \quad \frac{f(x + h) - f(x - h)}{2h}$$

## Key Objectives

1. Deriving derivatives from first principles ($x^2 + 2x$, $x^3$, $kx$, $1/x$).
2. Comparing forward vs. central difference convergence rates.
3. Analyzing the **truncation error vs. roundoff error** trade-off (catastrophic cancellation).
4. Implementing a production-quality gradient checker.


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = [7.0, 4.5]
plt.rcParams["figure.autolayout"] = True

## 1. Analytic Derivatives

Consider two benchmark polynomials:
1. $f_1(x) = x^2 + 2x \implies f_1'(x) = 2x + 2$
2. $f_2(x) = x^3 \implies f_2'(x) = 3x^2$


In [ ]:
def f1(x: float) -> float:
    return x**2 + 2.0 * x


def df1_exact(x: float) -> float:
    return 2.0 * x + 2.0


def f2(x: float) -> float:
    return x**3


def df2_exact(x: float) -> float:
    return 3.0 * (x**2)


# Verify analytical derivatives at x = 2.0
assert df1_exact(2.0) == 6.0
assert df2_exact(2.0) == 12.0
print(f"Exact f1'(2.0) = {df1_exact(2.0)}")
print(f"Exact f2'(2.0) = {df2_exact(2.0)}")

In [ ]:
# Visualization of f(x) and its derivative
x_domain = np.linspace(-4.0, 3.0, 200)
plt.figure()
plt.plot(x_domain, [f1(x) for x in x_domain], "b-", label=r"$f(x) = x^2 + 2x$")
plt.plot(x_domain, [df1_exact(x) for x in x_domain], "r--", label=r"$f'(x) = 2x + 2$")
plt.plot([2.0], [f1(2.0)], "bo", label="Evaluation point (2, 8)")
plt.title("Function and its Analytic Derivative")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Finite Difference Implementations

We implement both the single-sided (forward) difference and the symmetric (central) difference.


In [ ]:
def forward_difference(f, x: float, h: float) -> float:
    """Compute forward difference: (f(x + h) - f(x)) / h."""
    return (f(x + h) - f(x)) / h


def central_difference(f, x: float, h: float) -> float:
    """Compute central difference: (f(x + h/2) - f(x - h/2)) / h."""
    return (f(x + h / 2.0) - f(x - h / 2.0)) / h


def central_difference_2h(f, x: float, h: float) -> float:
    """Alternative central difference: (f(x + h) - f(x - h)) / (2 * h)."""
    return (f(x + h) - f(x - h)) / (2.0 * h)

## 3. Empirical Comparison: Forward vs. Central

We evaluate both methods on $f_1(x) = x^2 + 2x$ at $x = 2$ ($f'(2) = 6.0$) across varying step sizes $h$.


In [ ]:
h_values = [1.0e2, 1.0e1, 1.0e0, 1.0e-1, 1.0e-2, 1.0e-3, 1.0e-4, 1.0e-5]

print("Evaluating f(x) = x^2 + 2x at x = 2.0 (Exact = 6.0):")
print(
    f"{'Step h':>10} | {'Forward Diff':>14} | {'Central Diff':>14} | {'Err Forward':>12} | {'Err Central':>12}"
)
print("-" * 75)

for h in h_values:
    fwd = forward_difference(f1, 2.0, h)
    cnt = central_difference(f1, 2.0, h)
    err_fwd = abs(fwd - 6.0)
    err_cnt = abs(cnt - 6.0)
    print(f"{h:10.1e} | {fwd:14.7f} | {cnt:14.7f} | {err_fwd:12.2e} | {err_cnt:12.2e}")

Notice that for $f(x) = x^2 + 2x$, the central difference produces an error of **zero** (up to float precision) for any moderate $h$!

Why? By Taylor series:
$$f(x + h/2) = f(x) + \frac{h}{2}f'(x) + \frac{h^2}{8}f''(x) + \frac{h^3}{48}f'''(x) + \dots$$
$$f(x - h/2) = f(x) - \frac{h}{2}f'(x) + \frac{h^2}{8}f''(x) - \frac{h^3}{48}f'''(x) + \dots$$

Subtracting the two equations cancels all **even-order terms** ($f(x)$, $f''(x)$, etc.):
$$\frac{f(x + h/2) - f(x - h/2)}{h} = f'(x) + \frac{h^2}{24}f'''(x) + \mathcal{O}(h^4)$$

Since $f_1(x) = x^2 + 2x$ is a quadratic polynomial, its third derivative $f_1'''(x) = 0$, meaning the central difference truncation error is **identically zero**!

Now let's examine $f_2(x) = x^3$, where $f_2'''(x) = 6 \neq 0$:


In [ ]:
print("Evaluating f(x) = x^3 at x = 2.0 (Exact = 12.0):")
print(
    f"{'Step h':>10} | {'Forward Diff':>14} | {'Central Diff':>14} | {'Err Forward':>12} | {'Err Central':>12}"
)
print("-" * 75)

for h in h_values:
    fwd = forward_difference(f2, 2.0, h)
    cnt = central_difference(f2, 2.0, h)
    err_fwd = abs(fwd - 12.0)
    err_cnt = abs(cnt - 12.0)
    print(f"{h:10.1e} | {fwd:14.7f} | {cnt:14.7f} | {err_fwd:12.2e} | {err_cnt:12.2e}")

Observe the convergence rates:
- As $h$ decreases by a factor of 10, the forward difference error decreases by **10** (first-order $\mathcal{O}(h)$).
- The central difference error decreases by **100** (second-order $\mathcal{O}(h^2)$).


## 4. The Step Size Trade-Off: Truncation vs. Catastrophic Cancellation

If smaller $h$ is mathematically better, can we simply choose $h = 10^{-20}$?

**No.** Floating-point numbers have finite precision (53 bits $\approx 15-17$ significant decimal digits in IEEE 754 float64). When $h$ is extremely small:
1. $x + h$ and $x$ become nearly identical, causing **subtractive cancellation**.
2. Dividing the noisy difference by an tiny $h$ amplifies the roundoff error catastrophically.


In [ ]:
h_scan = np.logspace(1, -18, 40)
errors_fwd = []
errors_cnt = []

for h in h_scan:
    fwd_val = forward_difference(f2, 2.0, h)
    cnt_val = central_difference(f2, 2.0, h)
    errors_fwd.append(max(1e-16, abs(fwd_val - 12.0)))
    errors_cnt.append(max(1e-16, abs(cnt_val - 12.0)))

plt.figure(figsize=(8, 5))
plt.loglog(h_scan, errors_fwd, "r.-", label="Forward Difference (O(h))")
plt.loglog(h_scan, errors_cnt, "b.-", label=r"Central Difference ($O(h^2)$)")
plt.axvline(1e-5, color="green", linestyle="--", alpha=0.7, label="Optimal step h ≈ 1e-5")
plt.title("Numerical Differentiation Error vs. Step Size h")
plt.xlabel("Step size h")
plt.ylabel("Absolute Error |f'_approx - f'_exact|")
plt.gca().invert_xaxis()
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

## 5. Robust Gradient Checker for Machine Learning

When implementing custom layers or autograd operators in deep learning, a **gradient check** compares the analytical backpropagation gradient with a high-precision central difference approximation. Relative error is computed as:

$$\text{rel\_error} = \frac{|g_{\text{analytic}} - g_{\text{numerical}}|}{\max(|g_{\text{analytic}}|, |g_{\text{numerical}}|) + \epsilon}$$


In [ ]:
def gradient_check(f, df_analytic, x: float, h: float = 1e-5, tol: float = 1e-6) -> bool:
    """Perform gradient verification using central difference."""
    g_num = (f(x + h) - f(x - h)) / (2.0 * h)
    g_ana = df_analytic(x)
    rel_error = abs(g_ana - g_num) / (max(abs(g_ana), abs(g_num)) + 1e-12)
    print(
        f"Gradient check at x={x}: Analytic={g_ana:.8f}, Numerical={g_num:.8f}, RelError={rel_error:.2e}"
    )
    return rel_error < tol


# Test across multiple functions
assert gradient_check(math.sin, math.cos, x=1.0)
assert gradient_check(lambda x: x**4 - 3 * x**2, lambda x: 4 * x**3 - 6 * x, x=2.5)
assert gradient_check(math.exp, math.exp, x=0.5)
print("All gradient checks passed successfully!")

## Summary

- The central difference formula provides second-order $\mathcal{O}(h^2)$ accuracy due to symmetric cancellation of odd Taylor series terms.
- The optimal step size in double-precision arithmetic is typically $h \approx 10^{-5}$ to $10^{-8}$. Step sizes below $10^{-14}$ suffer from catastrophic cancellation.
- Gradient checking is an essential debugging technique for validating manual backward passes in PyTorch or NumPy neural network implementations.
